In [ ]:
import pandas as pd
import numpy as np
import re, json
import xgboost as xgb
from IPython.display import display

train = pd.read_csv("/content/dsp_knob_variant_dataset_alltracks_train.csv")
val   = pd.read_csv("/content/dsp_knob_variant_dataset_alltracks_val.csv")
test  = pd.read_csv("/content/dsp_knob_variant_dataset_alltracks_test.csv")
ctrl  = pd.read_csv("/content/controller_dataset_yamnet_m003.csv")

print(train.shape, val.shape, test.shape, ctrl.shape)

(3774, 20) (867, 20) (1173, 20) (114, 1051)


In [ ]:
def norm_track(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip()
    s = s.split("/")[-1].split("\\")[-1]
    s = re.sub(r"\.(wav|mp3|flac|m4a|ogg)$", "", s, flags=re.IGNORECASE)
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s)
    return s.lower().strip()

ctrl = ctrl.copy()
ctrl["track_key"] = ctrl["track"].map(norm_track)

def merge_ctrl(df):
    df = df.copy()
    df["track_key"] = df["track"].map(norm_track)
    return df.merge(ctrl.drop(columns=["track"]), on="track_key", how="left")

tr = merge_ctrl(train)
va = merge_ctrl(val)
te = merge_ctrl(test)

print("merged:", tr.shape, va.shape, te.shape)

merged: (3774, 1071) (867, 1071) (1173, 1071)


In [ ]:
def build_hardneg(df, k_top=15, k_rand=15, seed=1337):
    rng = np.random.default_rng(seed)
    out = []
    for t, g in df.groupby("track"):
        g = g.sort_values("score", ascending=False).copy()
        topk = g.head(k_top)
        rest = g.iloc[k_top:]
        if len(rest) > 0:
            take_n = min(k_rand, len(rest))
            rand_idx = rng.choice(rest.index.to_numpy(), size=take_n, replace=False)
            rand = rest.loc[rand_idx]
            keep = pd.concat([topk, rand], axis=0)
        else:
            keep = topk
        out.append(keep.drop_duplicates())
    return pd.concat(out, axis=0).reset_index(drop=True)

tr_h = build_hardneg(tr, 15, 15)
va_h = build_hardneg(va, 15, 15)
te_h = build_hardneg(te, 15, 15)

print("After:", len(tr_h), len(va_h), len(te_h))
print("Avg/train candidates:", len(tr_h)/tr_h["track"].nunique())

After: 2220 510 690
Avg/train candidates: 30.0


In [ ]:
def add_relevance(df, n_levels=5):
    out = []
    for t, g in df.groupby("track"):
        g = g.copy()
        g["rank"] = g["score"].rank(method="average", ascending=True)
        r = (g["rank"] - 1) / max(len(g) - 1, 1)
        g["rel"] = np.floor(r * (n_levels - 1 + 1e-9)).astype(int)
        out.append(g.drop(columns=["rank"]))
    return pd.concat(out, axis=0).reset_index(drop=True)

trR = add_relevance(tr_h, 5)
vaR = add_relevance(va_h, 5)

print(trR["rel"].value_counts().sort_index())

rel
0    592
1    518
2    519
3    517
4     74
Name: count, dtype: int64


In [ ]:
NON_FEATURE = {
    "track", "variant_id",
    "label_is_best", "label_apply_dsp_track",
    "focus", "sim", "score",
    "best_score", "second_score", "gap_best_second",
    "is_ambiguous", "is_low_sim_track",
    "rel",
}

def infer_feature_cols(df):
    num_cols = []
    df = df.copy()
    for c in df.columns:
        if c in NON_FEATURE:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            num_cols.append(c)
        else:
            tmp = pd.to_numeric(df[c], errors="coerce")
            if tmp.notna().mean() > 0.95:
                df[c] = tmp
                num_cols.append(c)
    return sorted(num_cols)

feature_cols = infer_feature_cols(tr_h)
print("Num features:", len(feature_cols))
print("First 20:", feature_cols[:20])
assert len(feature_cols) > 0

Num features: 1055
First 20: ['best_compressor_strength', 'best_drc_strength', 'best_grid_focus', 'best_grid_score', 'best_grid_sim', 'best_high_gain_db', 'best_low_gain_db', 'best_mid_gain_db', 'best_transient_amount', 'best_transient_smooth', 'centroid', 'delta', 'drc_strength', 'flatness', 'focus_base', 'focus_instr', 'focus_lofi', 'high_gain_db', 'instr_score', 'low_gain_db']


In [ ]:
def make_rank_dmatrix(df, feature_cols, label_col="rel"):
    df_sorted = df.sort_values(["track"]).reset_index(drop=True)
    X = df_sorted[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
    y = df_sorted[label_col].astype(int).values
    group_sizes = df_sorted.groupby("track").size().to_numpy()
    dmat = xgb.DMatrix(X, label=y)
    dmat.set_group(group_sizes)
    return df_sorted, dmat

train_sorted, dtrain = make_rank_dmatrix(trR, feature_cols, "rel")
val_sorted, dval     = make_rank_dmatrix(vaR, feature_cols, "rel")

In [ ]:
params = {
    "objective": "rank:ndcg",
    "eval_metric": "ndcg@1",
    "eta": 0.05,
    "max_depth": 6,
    "subsample": 0.9,
    "colsample_bytree": 0.7,
    "lambda": 2.0,
    "seed": 1337,
}

ranker = xgb.train(
    params,
    dtrain,
    num_boost_round=1200,
    evals=[(dtrain,"train"), (dval,"val")],
    early_stopping_rounds=50,
    verbose_eval=50
)

[0]	train-ndcg@1:0.47297	val-ndcg@1:0.49412
[50]	train-ndcg@1:0.97838	val-ndcg@1:0.89020
[85]	train-ndcg@1:0.98559	val-ndcg@1:0.85882


In [ ]:
def controller_eval_with_preds(df_full, preds, name):
    df = df_full.copy().reset_index(drop=True)
    df["pred_score"] = preds
    idx = df.groupby("track")["pred_score"].idxmax()
    chosen = df.loc[idx].copy()

    best_idx = df.groupby("track")["score"].idxmax()
    best = df.loc[best_idx][["track","variant_id","score"]].rename(
        columns={"variant_id":"true_variant_id","score":"true_best_score"}
    )

    out = chosen.merge(best, on="track", how="left")
    out["hit"] = (out["variant_id"] == out["true_variant_id"])
    out["regret"] = out["true_best_score"] - out["score"]
    print(f"{name} tracks:", out["track"].nunique())
    print(f"{name} Top-1 hit:", out["hit"].mean())
    print(f"{name} regret mean={out['regret'].mean():.4f} median={out['regret'].median():.4f} max={out['regret'].max():.4f}")
    display(out.sort_values("regret", ascending=False).head(20))
    return out

val_preds  = ranker.predict(xgb.DMatrix(va[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values))
test_preds = ranker.predict(xgb.DMatrix(te[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values))

val_report  = controller_eval_with_preds(va, val_preds, "VAL ranker (full grid)")
test_report = controller_eval_with_preds(te, test_preds, "TEST ranker (full grid)")

VAL ranker (full grid) tracks: 17
VAL ranker (full grid) Top-1 hit: 0.7647058823529411
VAL ranker (full grid) regret mean=0.0377 median=0.0000 max=0.4867


,track,variant_id,label_is_best,label_apply_dsp_track,focus,sim,score,focus_instr,sim_instr,instr_score,...,yamnet_1019,yamnet_1020,yamnet_1021,yamnet_1022,yamnet_1023,pred_score,true_variant_id,true_best_score,hit,regret
9,Katy Kirby - Come Back to Nashville,l-3.0_m-6.0_h-3.0_ts0.00_drc0.40,0,1,4.773206e-03,0.908352,0.275847,6.270960e-03,0.891098,0.271719,...,0.000000,0.039879,0.060211,0.049215,0.000000,0.574793,l-3.0_m-6.0_h+0.0_ts0.00_drc0.80,0.762541,False,0.486694
3,Clan Destined - Never All Ways (clean),l-3.0_m-6.0_h-3.0_ts0.00_drc0.40,0,1,6.751086e-01,0.625059,0.660094,1.587918e-01,0.661093,0.309482,...,0.000000,0.016472,0.008760,0.024485,0.000080,0.649569,l-3.0_m-6.0_h-6.0_ts0.00_drc0.80,0.803699,False,0.143605
11,Shy and the Fight - Here the sea speaks in whi...,l+3.0_m-6.0_h-3.0_ts0.00_drc0.00,0,1,2.434662e-01,0.851491,0.425874,8.647840e-02,0.888642,0.327128,...,0.000000,0.055225,0.064425,0.525503,0.000000,1.119943,l+3.0_m-6.0_h+0.0_ts0.00_drc0.00,0.435509,False,0.009635
10,Philippa Dowding and Allister Thompson - Winter,l+3.0_m-6.0_h-3.0_ts0.00_drc0.00,0,1,1.406752e-02,0.986817,0.305892,1.217715e-03,0.999319,0.300648,...,0.000000,0.051132,0.067225,0.145644,0.000000,1.024492,l+3.0_m-6.0_h+0.0_ts0.00_drc0.00,0.306665,False,0.000773
0,99 Tales - Baby Out Of Jail,l-3.0_m-3.0_h+0.0_ts0.00_drc0.00,1,1,1.590474e-07,0.938597,0.281579,1.605836e-07,0.934535,0.280361,...,0.000000,0.053369,0.131881,0.129837,0.000000,1.020866,l-3.0_m-3.0_h+0.0_ts0.00_drc0.00,0.281579,True,0.000000
5,Dengaz - FayaFlow (feat. Praga),l+0.0_m+0.0_h-6.0_ts0.00_drc0.00,1,1,8.179902e-07,0.838575,0.251573,2.959166e-07,0.826739,0.248022,...,0.000000,0.029294,0.108805,0.055651,0.000000,1.031639,l+0.0_m+0.0_h-6.0_ts0.00_drc0.00,0.251573,True,0.000000
1,A. Cooper - Support,l+3.0_m-6.0_h-6.0_ts0.30_drc0.00,1,1,9.904647e-01,0.824335,0.940626,3.385015e-01,0.994976,0.535444,...,0.000000,0.161932,0.041655,0.199276,0.000000,0.285234,l+3.0_m-6.0_h-6.0_ts0.30_drc0.00,0.940626,True,0.000000
2,Breuss Arrizabalaga Quintet - Tsurugi,l-3.0_m+0.0_h+0.0_ts0.00_drc0.00,1,1,4.339128e-03,0.994529,0.301396,2.679290e-03,0.994954,0.300362,...,0.000000,0.005227,0.031007,0.055854,0.000000,0.739087,l-3.0_m+0.0_h+0.0_ts0.00_drc0.00,0.301396,True,0.000000
4,Cletus Got Shot - Charlie Jones,l-3.0_m-6.0_h+0.0_ts0.00_drc0.00,1,1,8.784346e-06,0.957827,0.287354,4.715313e-06,0.951424,0.285430,...,0.000000,0.034995,0.088709,0.131760,0.000000,0.970280,l-3.0_m-6.0_h+0.0_ts0.00_drc0.00,0.287354,True,0.000000
8,Just-Release-Me-MASTER38(chosic.com),l-3.0_m-3.0_h-6.0_ts0.00_drc0.00,1,1,7.222738e-04,0.956144,0.287349,2.175486e-04,0.950900,0.285422,...,0.000000,0.067342,0.057954,0.256622,0.000000,1.081791,l-3.0_m-3.0_h-6.0_ts0.00_drc0.00,0.287349,True,0.000000


TEST ranker (full grid) tracks: 23
TEST ranker (full grid) Top-1 hit: 0.782608695652174
TEST ranker (full grid) regret mean=0.0262 median=0.0000 max=0.3429


,track,variant_id,label_is_best,label_apply_dsp_track,focus,sim,score,focus_instr,sim_instr,instr_score,...,yamnet_1019,yamnet_1020,yamnet_1021,yamnet_1022,yamnet_1023,pred_score,true_variant_id,true_best_score,hit,regret
18,Thorn & Shout - Line & Lure,l-3.0_m+0.0_h-6.0_ts0.00_drc0.00,0,1,1.878198e-03,0.930648,0.280509,4.703000e-04,0.935152,0.280875,...,0.000000,0.153086,0.103055,0.057065,0.000000,0.348821,l-3.0_m+0.0_h-6.0_ts0.60_drc0.00,0.623366,False,0.342857
3,Dead Times - Slow Burning Bliss,l-3.0_m+0.0_h+0.0_ts0.00_drc0.00,0,1,4.006365e-02,0.962299,0.316734,2.993093e-02,0.963101,0.309882,...,0.000000,0.223487,0.040004,0.111459,0.000000,1.107636,l-3.0_m+0.0_h-6.0_ts0.00_drc0.00,0.441885,False,0.125150
12,Nexto - Catwalk ft. DMac YSD,l-3.0_m-6.0_h+0.0_ts0.00_drc0.40,0,1,7.885928e-01,0.694700,0.760425,5.113932e-01,0.732521,0.577731,...,0.000000,0.074827,0.019283,0.048174,0.000003,0.610180,l+3.0_m-6.0_h-6.0_ts0.00_drc0.80,0.885197,False,0.124772
16,SK - Runnin out of time,l-3.0_m-3.0_h+0.0_ts0.00_drc0.40,0,1,9.796729e-01,0.656249,0.882646,7.983708e-01,0.699153,0.768605,...,0.000000,0.051810,0.008659,0.073294,0.000031,0.414028,l+3.0_m-3.0_h-6.0_ts0.00_drc0.80,0.888102,False,0.005456
13,Nexto - Show Some Love (It's Christmas),l-3.0_m+0.0_h-3.0_ts0.00_drc0.40,0,1,9.947819e-01,0.698647,0.905941,9.830472e-01,0.717067,0.903253,...,0.000000,0.289947,0.008501,0.211803,0.000000,0.567938,l+3.0_m+0.0_h-6.0_ts0.00_drc0.40,0.909208,False,0.003267
1,Andrea - Work the Middle (Real Remix),l+3.0_m-6.0_h-6.0_ts0.00_drc0.00,1,1,9.213372e-01,0.933524,0.924993,4.191344e-01,0.974913,0.585868,...,0.000000,0.050771,0.011906,0.299089,0.000000,1.085665,l+3.0_m-6.0_h-6.0_ts0.00_drc0.00,0.924993,True,0.000000
0,Alaclair Ensemble - LICORNES,l+0.0_m+0.0_h+0.0_ts0.00_drc0.40,1,1,9.951574e-01,0.842400,0.949330,9.854769e-01,0.835888,0.940600,...,0.000000,0.138247,0.007606,0.092338,0.000000,0.784977,l+0.0_m+0.0_h+0.0_ts0.00_drc0.40,0.949330,True,0.000000
6,Josh Woodward - Omaha,l-3.0_m+0.0_h+0.0_ts0.00_drc0.00,1,1,1.729336e-03,0.953246,0.287184,8.658750e-04,0.954358,0.286914,...,0.000000,0.145436,0.189872,0.162678,0.000000,1.313042,l-3.0_m+0.0_h+0.0_ts0.00_drc0.00,0.287184,True,0.000000
5,Jasmine Jordan - Closer,l+3.0_m-6.0_h-6.0_ts0.00_drc0.00,1,1,9.492839e-01,0.775332,0.897098,6.406204e-01,0.773054,0.680351,...,0.000000,0.232695,0.019327,0.453991,0.000000,1.237901,l+3.0_m-6.0_h-6.0_ts0.00_drc0.00,0.897098,True,0.000000
4,Ignatz - Rise and Fall,l-3.0_m-3.0_h+0.0_ts0.00_drc0.00,1,1,8.777292e-04,0.998654,0.300211,5.949569e-04,0.999037,0.300128,...,0.000000,0.263547,0.288285,0.004817,0.000000,0.657542,l-3.0_m-3.0_h+0.0_ts0.00_drc0.00,0.300211,True,0.000000


In [ ]:
def topk_hit_rate(df_full, preds, k=3):
    df = df_full.copy().reset_index(drop=True)
    df["pred"] = preds

    best_idx = df.groupby("track")["score"].idxmax()
    best = df.loc[best_idx][["track","variant_id"]].rename(columns={"variant_id":"true_best"})

    hits = []
    for t, g in df.groupby("track"):
        topk = g.sort_values("pred", ascending=False).head(k)
        true_best = best.loc[best["track"] == t, "true_best"].iloc[0]
        hits.append(true_best in set(topk["variant_id"]))
    return float(np.mean(hits))

print("VAL top-3 hit:", topk_hit_rate(va, val_preds, 3))
print("VAL top-5 hit:", topk_hit_rate(va, val_preds, 5))
print("TEST top-3 hit:", topk_hit_rate(te, test_preds, 3))
print("TEST top-5 hit:", topk_hit_rate(te, test_preds, 5))

VAL top-3 hit: 0.9411764705882353
VAL top-5 hit: 1.0
TEST top-3 hit: 0.9565217391304348
TEST top-5 hit: 0.9565217391304348


In [ ]:
ranker.save_model("xgb_ranker_controller.json")
with open("controller_feature_cols.json", "w") as f:
    json.dump(feature_cols, f)

print("Saved model + feature list:", len(feature_cols), "features")

Saved model + feature list: 1055 features


In [ ]:
import json, xgboost as xgb, pandas as pd, numpy as np

def pick_best_variant_per_track(df_full, model, feature_cols):
    X = df_full[feature_cols].astype(float).values
    preds = model.predict(xgb.DMatrix(X))
    df = df_full.copy().reset_index(drop=True)
    df["pred"] = preds
    idx = df.groupby("track")["pred"].idxmax()
    chosen = df.loc[idx].copy()
    return chosen[["track","variant_id","low_gain_db","mid_gain_db","high_gain_db","transient_smooth","drc_strength","pred","score"]]

chosen_test = pick_best_variant_per_track(te, ranker, feature_cols)
chosen_val  = pick_best_variant_per_track(va, ranker, feature_cols)

chosen_test.to_csv("chosen_knobs_test.csv", index=False)
chosen_val.to_csv("chosen_knobs_val.csv", index=False)

print("Saved chosen_knobs_val.csv and chosen_knobs_test.csv")
chosen_test.head(10)

Saved chosen_knobs_val.csv and chosen_knobs_test.csv


,track,variant_id,low_gain_db,mid_gain_db,high_gain_db,transient_smooth,drc_strength,pred,score
1020,Alaclair Ensemble - LICORNES,l+0.0_m+0.0_h+0.0_ts0.00_drc0.40,0.0,0.0,0.0,0.0,0.4,0.784977,0.949330
306,Andrea - Work the Middle (Real Remix),l+3.0_m-6.0_h-6.0_ts0.00_drc0.00,3.0,-6.0,-6.0,0.0,0.0,1.085665,0.924993
1122,Beat Mekanik - Harley Davidson,l+0.0_m+0.0_h+0.0_ts0.00_drc0.00,0.0,0.0,0.0,0.0,0.0,1.170074,0.299639
559,Dead Times - Slow Burning Bliss,l-3.0_m+0.0_h+0.0_ts0.00_drc0.00,-3.0,0.0,0.0,0.0,0.0,1.107636,0.316734
408,Ignatz - Rise and Fall,l-3.0_m-3.0_h+0.0_ts0.00_drc0.00,-3.0,-3.0,0.0,0.0,0.0,0.657542,0.300211
612,Jasmine Jordan - Closer,l+3.0_m-6.0_h-6.0_ts0.00_drc0.00,3.0,-6.0,-6.0,0.0,0.0,1.237901,0.897098
969,Josh Woodward - Omaha,l-3.0_m+0.0_h+0.0_ts0.00_drc0.00,-3.0,0.0,0.0,0.0,0.0,1.313042,0.287184
816,Litmus - This Is The Place,l-3.0_m+0.0_h-6.0_ts0.30_drc0.00,-3.0,0.0,-6.0,0.3,0.0,0.936267,0.759713
153,Luke-Bergs-We-Made-It(chosic.com),l+3.0_m-3.0_h-6.0_ts0.00_drc0.00,3.0,-3.0,-6.0,0.0,0.0,0.939566,0.853141
765,Malaventura_-_12_-_The_Queen(chosic.com),l-3.0_m-3.0_h-3.0_ts0.00_drc0.00,-3.0,-3.0,-3.0,0.0,0.0,0.855401,0.999623


In [ ]:
from google.colab import files

files.download("xgb_ranker_controller.json")
files.download("controller_feature_cols.json")
files.download("chosen_knobs_val.csv")
files.download("chosen_knobs_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>